In [1]:
import requests
import concurrent.futures as cf
import time
import pandas as pd
import selenium
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import lxml.html as lx
from lxml import etree
import re
from datetime import date, timedelta


headers = {
    'User - agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:142.0) Gecko/20100101 Firefox/142.0',
}

In [4]:
# One case scenario
base_url = "https://cityofdavis.eco-counter.us/?"
index_url = 'https://cityofdavis.eco-counter.us/?startDate=2024-01-01&endDate=2024-01-01'

In [5]:
# Not accelerated code. Next chunk speeds it up.
def daterange(start_date: date, end_date: date):
    days = int((end_date - start_date).days)
    for n in range(days):
        yield start_date + timedelta(n)

def selenium_scrape(single_date):
    current_page = base_url + "startDate=" + single_date.strftime("%Y-%m-%d") + "&endDate=" + single_date.strftime("%Y-%m-%d")

    option = webdriver.ChromeOptions()
    
    with webdriver.Chrome(options=option) as driver:
        driver.implicitly_wait(10)
        driver.get(current_page)
        string = driver.find_element(By.XPATH, "//p[@data-testid='undefined-value']").text
        cleaned_string = string.replace(',', '')
        daily_count = int(cleaned_string)
        print(f"{single_date}: {daily_count}")

    return pd.DataFrame({'Date': [single_date], 'Count': [daily_count]})

In [7]:
start_date = date(2024, 1, 1)
end_date = date(2026, 1, 1)
dataframes = []
range_dates = list(daterange(start_date, end_date))

with cf.ThreadPoolExecutor(max_workers=5) as executor:
    counts = executor.map(selenium_scrape, range_dates)

    for count in counts:
        dataframes.append(count)


table = pd.concat(dataframes, ignore_index = True)                

2024-01-03: 475
2024-01-05: 646
2024-01-04: 580
2024-01-01: 307
2024-01-02: 360
2024-01-09: 2374
2024-01-07: 1013
2024-01-10: 2428
2024-01-08: 2396
2024-01-06: 448
2024-01-11: 2459
2024-01-13: 724
2024-01-14: 1055
2024-01-15: 1115
2024-01-12: 2097
2024-01-20: 767
2024-01-16: 1663
2024-01-19: 1994
2024-01-18: 2470
2024-01-17: 2345
2024-01-21: 939
2024-01-22: 1708
2024-01-23: 2354
2024-01-25: 2380
2024-01-24: 1411
2024-01-26: 2143
2024-01-27: 1718
2024-01-28: 1581
2024-01-30: 2524
2024-01-29: 2257
2024-01-31: 1224
2024-02-01: 2183
2024-02-02: 2069
2024-02-03: 1494
2024-02-04: 394
2024-02-06: 2365
2024-02-07: 1165
2024-02-05: 1831
2024-02-08: 2138
2024-02-09: 2026
2024-02-10: 1420
2024-02-11: 1134
2024-02-12: 1999
2024-02-14: 1743
2024-02-13: 2237
2024-02-15: 2183
2024-02-16: 1879
2024-02-17: 593
2024-02-19: 453
2024-02-18: 780
2024-02-21: 2032
2024-02-22: 2219
2024-02-20: 1681
2024-02-23: 2210
2024-02-24: 1893
2024-02-25: 1503
2024-02-26: 2079
2024-02-28: 2240
2024-02-29: 1613
2024-02-27

In [8]:
table.to_csv('ecocounter_bike_traffic_data.csv', index = False)

Acknowledgements

In [9]:
https://stackoverflow.com/questions/1060279/iterating-through-a-range-of-dates-in-python

SyntaxError: invalid syntax (2022555073.py, line 1)